In [1]:
import os
import glob
import numpy as np
import pandas as pd
import networkx as nx
from scipy.linalg import pinv, eigh
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
# =====================================================================
# Module 1: Mathematical Engine (Arakelov-Green Spectral Extractor)
# =====================================================================
class TropicalGeometricExtractor:
    def __init__(self, nx_tree: nx.Graph, nodes_coords: dict, root_node: int):
        self.G = nx_tree.copy()
        self.nodes_info = nodes_coords
        self.root_node = root_node
        self._skeletonize_graph()
        self.node_list = list(self.G.nodes())
        self.node_to_idx = {n: i for i, n in enumerate(self.node_list)}
        
    def _skeletonize_graph(self):
        """Reduces degree-2 nodes to compact the graph structure."""
        deg2_nodes = [n for n in self.G.nodes() if self.G.degree(n) == 2 and n != self.root_node]
        for n in deg2_nodes:
            neighbors = list(self.G.neighbors(n))
            if len(neighbors) == 2:
                u, v = neighbors
                new_w = self.G[u][n]['weight'] + self.G[n][v]['weight']
                if self.G.has_edge(u, v):
                    self.G[u][v]['weight'] = min(self.G[u][v]['weight'], new_w)
                else:
                    self.G.add_edge(u, v, weight=new_w)
                self.G.remove_node(n)

    def get_spectral_signature(self, epsilon: float = 20.0, tau: float = 5.0, top_k: int = 64) -> np.ndarray:
        """Extracts topological invariants using the Arakelov-Green pseudoinverse measure."""
        leaves = [n for n, d in self.G.degree() if d == 1 and n != self.root_node]
        for leaf in leaves:
            dist = np.linalg.norm(self.nodes_info[leaf] - self.nodes_info[self.root_node])
            if dist < epsilon:
                self.G.add_edge(leaf, self.root_node, weight=dist)

        bridges = set(nx.bridges(self.G))
        n = len(self.node_list)
        if n < 2: return np.zeros(top_k)
        
        W = np.zeros((n, n))
        for u, v, data in self.G.edges(data=True):
            i, j = self.node_to_idx[u], self.node_to_idx[v]
            is_noise = ((u, v) in bridges or (v, u) in bridges) and data['weight'] < tau
            
            # Rigorous zero-division protection
            weight = max(data['weight'], 1e-5) 
            W[i, j] = W[j, i] = 1e9 if is_noise else (1.0 / weight)
            
        D = np.diag(np.sum(W, axis=1))
        L_pinv = pinv(D - W)
        diag = np.diag(L_pinv)
        D_AG = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
        
        eigenvalues = np.sort(np.abs(eigh(D_AG, eigvals_only=True)))[::-1]
        sig = np.zeros(top_k)
        actual_k = min(top_k, len(eigenvalues))
        sig[:actual_k] = eigenvalues[:actual_k]
        return sig


# =====================================================================
# Module 2: MorphoGNN Core Operators (EdgeConv & KNN)
# =====================================================================
def knn(x: torch.Tensor, k: int) -> torch.Tensor:
    """
    K-Nearest Neighbors search for point clouds.
    Input x tensor shape: (Batch, 3, N)
    """
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]   # (B, N, k)
    return idx

def get_graph_feature(x: torch.Tensor, k: int = 16, idx: torch.Tensor = None) -> torch.Tensor:
    """Extracts local neighborhood geometric features (directional vectors)."""
    batch_size, _, num_points = x.size()
    if idx is None:
        idx = knn(x, k=k)
        
    device = x.device
    idx_base = torch.arange(0, batch_size, device=device).view(-1, 1, 1) * num_points
    idx = idx + idx_base
    idx = idx.view(-1)
    
    _, num_dims, _ = x.size()
    x = x.transpose(2, 1).contiguous()
    feature = x.view(batch_size * num_points, -1)[idx, :]
    feature = feature.view(batch_size, num_points, k, num_dims) 
    x = x.view(batch_size, num_points, 1, num_dims).repeat(1, 1, k, 1)
    
    feature = torch.cat((feature - x, x), dim=3).permute(0, 3, 1, 2).contiguous()
    return feature

# =====================================================================
# Module 3: Tropical-MorphoGNN Fusion Architecture
# =====================================================================
class TropicalMorphoGNN(nn.Module):
    def __init__(self, num_classes: int, k: int = 16, spectral_dim: int = 64):
        super().__init__()
        self.k = k
        
        # --- MorphoGNN Point Cloud Backbone ---
        self.conv1 = nn.Sequential(nn.Conv2d(6, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(64, 64, kernel_size=1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, kernel_size=1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(256, 256, kernel_size=1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(256, 1024, kernel_size=1, bias=False), nn.BatchNorm1d(1024), nn.LeakyReLU(0.2))
        
        # --- Tropical Algebraic Geometry Prior Projection ---
        self.spectral_proj = nn.Sequential(
            nn.Linear(spectral_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        
        # --- Late Fusion & Classifier (2048 + 128 = 2176) ---
        self.linear1 = nn.Linear(2048 + 128, 512, bias=False)
        self.bn6 = nn.BatchNorm1d(512)
        self.dp1 = nn.Dropout(p=0.5)
        self.linear2 = nn.Linear(512, 256)
        self.bn7 = nn.BatchNorm1d(256)
        self.dp2 = nn.Dropout(p=0.5)
        self.linear3 = nn.Linear(256, num_classes)

    def forward(self, x_pc: torch.Tensor, x_spec: torch.Tensor) -> torch.Tensor:
        """
        x_pc: (B, 3, 1024) Normalized and centered point cloud coordinates
        x_spec: (B, 64) Absolute mathematical feature spectrum
        """
        batch_size = x_pc.size(0)
        
        # Local point cloud EdgeConv operations
        x = get_graph_feature(x_pc, k=self.k)
        x = self.conv1(x).max(dim=-1, keepdim=False)[0]
        x = get_graph_feature(x, k=self.k)
        x = self.conv2(x).max(dim=-1, keepdim=False)[0]
        x = get_graph_feature(x, k=self.k)
        x = self.conv3(x).max(dim=-1, keepdim=False)[0]
        x = get_graph_feature(x, k=self.k)
        x = self.conv4(x).max(dim=-1, keepdim=False)[0]
        
        x = self.conv5(x)
        
        # MorphoGNN Double Pooling Strategy
        x_max = F.adaptive_max_pool1d(x, 1).view(batch_size, -1)
        x_avg = F.adaptive_avg_pool1d(x, 1).view(batch_size, -1)
        h_pointcloud = torch.cat((x_max, x_avg), dim=1) # (B, 2048)
        
        # Late Fusion: Concatenating spatial texture with global topological tension
        h_spectral = self.spectral_proj(x_spec) # (B, 128)
        h_fused = torch.cat([h_pointcloud, h_spectral], dim=-1) # (B, 2176)
        
        # Classification
        x = F.leaky_relu(self.bn6(self.linear1(h_fused)), negative_slope=0.2)
        x = self.dp1(x)
        x = F.leaky_relu(self.bn7(self.linear2(x)), negative_slope=0.2)
        x = self.dp2(x)
        return self.linear3(x)

# =====================================================================
# Module 4: Point Cloud Dataset Feature Engineering
# =====================================================================
class MorphoPointDataset(torch.utils.data.Dataset):
    def __init__(self, dataset_name: str, swc_dir: str, csv_path: str, phase: str = "train", num_points: int = 1024):
        self.num_points = num_points
        valid_classes = {
            "ACT-4": ["Isocortex_layer23", "Isocortex_layer4", "Isocortex_layer5", "Isocortex_layer6"],
            "BIL-6": ["CP", "Isocortex_layer23", "Isocortex_layer4", "Isocortex_layer5", "Isocortex_layer6", "VPM"],
            "JML-4": ["Isocortex_layer23", "Isocortex_layer5", "Isocortex_layer6", "VPM"]
        }
        target_classes = valid_classes[dataset_name]
        self.label_map = {cls_name: i for i, cls_name in enumerate(target_classes)}
        
        df = pd.read_csv(csv_path)
        local_swc_files = glob.glob(os.path.join(swc_dir, "*.swc"))
        local_basenames = [os.path.basename(f) for f in local_swc_files]
        
        self.data_list = []
        print(f"[{phase.upper()}] Sampling {dataset_name} to {num_points} points and extracting mathematical spectra...")
        
        for _, row in tqdm(df.iterrows(), total=len(df)):
            label = str(row['structure_merge__acronym']).replace("2/3", "23")
            if label not in target_classes: continue
                
            fold = float(row['model__fold'])
            if phase == "train" and fold >= 8.0: continue
            if phase == "test" and fold < 8.0: continue
                
            fname_csv = str(row['swc__fname'])
            spec_id = str(row.get('specimen__id', ''))
            swc_path = next((lf for lf, l_base in zip(local_swc_files, local_basenames) 
                             if l_base == fname_csv or l_base == fname_csv.replace("_reg.swc", ".swc") or (spec_id and spec_id in l_base)), None)
            if not swc_path: continue

            G_tree, nodes_info, root_node = self._parse_swc(swc_path)
            if len(G_tree) < 2: continue
            
            # 1. Compute Tropical Algebraic Spectral Features (Preserves absolute physical scale)
            engine = TropicalGeometricExtractor(G_tree, nodes_info, root_node)
            spectral_sig = engine.get_spectral_signature(top_k=64)
            if np.sum(spectral_sig) == 0: continue
            
            # 2. Point Cloud Normalization & Sampling for MorphoGNN
            coords = np.array([nodes_info[n] for n in G_tree.nodes()])
            root_coord = nodes_info[root_node]
            
            # Orthogonal Feature Decoupling: Translation Invariance
            coords = coords - root_coord
            
            # Scale Decoupling: Unit Sphere Normalization
            # Absolute volume is retained in spectral_sig. The point cloud strictly models "shape".
            max_radius = np.max(np.linalg.norm(coords, axis=1)) + 1e-5
            coords = coords / max_radius 
            
            n_actual = coords.shape[0]
            if n_actual >= self.num_points:
                # Sufficient nodes: Downsampling without replacement. 
                # Apply standard point cloud jittering during training.
                indices = np.random.choice(n_actual, self.num_points, replace=False)
                sampled_coords = coords[indices, :]
                if phase == "train":
                    sampled_coords += np.random.normal(0, 1e-3, size=sampled_coords.shape)
            else:
                # Sparse nodes: Oversampling with replacement.
                # Gaussian jittering is mandatory here to break spatial symmetry 
                # and prevent KNN manifold collapse (Zero-Distance Paradox).
                indices = np.random.choice(n_actual, self.num_points, replace=True)
                sampled_coords = coords[indices, :]
                jitter = np.random.normal(0, 1e-3, size=sampled_coords.shape)
                sampled_coords += jitter
                
            self.data_list.append({
                'point_cloud': torch.tensor(sampled_coords, dtype=torch.float32).transpose(0, 1), # MorphoGNN requires (3, N)
                'spectral_sig': torch.tensor(spectral_sig, dtype=torch.float32),
                'label': self.label_map[label]
            })

    def _parse_swc(self, filepath: str):
        G = nx.Graph()
        nodes_info, edges = {}, []
        root_node = None
        with open(filepath, 'r') as f:
            for line in f:
                if not line.strip() or line.startswith('#'): continue
                parts = line.split()
                if len(parts) < 7: continue
                nid, x, y, z, pid = int(parts[0]), float(parts[2]), float(parts[3]), float(parts[4]), int(parts[6])
                nodes_info[nid] = np.array([x, y, z])
                G.add_node(nid)
                if pid == -1: root_node = nid
                else: edges.append((nid, pid))
        if root_node is None and len(G) > 0: root_node = list(G.nodes())[0]
        for u, v in edges:
            if u in nodes_info and v in nodes_info:
                G.add_edge(u, v, weight=max(np.linalg.norm(nodes_info[u]-nodes_info[v]), 1e-5))
        if len(G) > 0:
            lcc = max(nx.connected_components(G), key=len)
            G = G.subgraph(lcc).copy()
            if root_node not in G: root_node = list(G.nodes())[0]
        return G, nodes_info, root_node

    def __len__(self): return len(self.data_list)
    def __getitem__(self, idx): return self.data_list[idx]

def pointcloud_collate_fn(batch: list) -> dict:
    pc = torch.stack([b['point_cloud'] for b in batch])
    sig = torch.stack([b['spectral_sig'] for b in batch])
    lbl = torch.tensor([b['label'] for b in batch], dtype=torch.long)
    return {'point_cloud': pc, 'spectral_sig': sig, 'label': lbl}

In [6]:
# =====================================================================
# Module 5: Training Execution
# =====================================================================
def train_and_evaluate(model: nn.Module, train_loader: torch.utils.data.DataLoader, test_loader: torch.utils.data.DataLoader, device: torch.device, epochs: int = 150, patience: int = 25, ds_name: str = "") -> float:
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_acc = 0.0
    epochs_no_improve = 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"[{ds_name}] Epoch {epoch:03d}/{epochs}")
        for batch in pbar:
            pc = batch['point_cloud'].to(device)
            sig = batch['spectral_sig'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            logits = model(pc, sig)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        scheduler.step()
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in test_loader:
                pc = batch['point_cloud'].to(device)
                sig = batch['spectral_sig'].to(device)
                labels = batch['label'].to(device)
                
                logits = model(pc, sig)
                preds = torch.argmax(logits, dim=-1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                
        acc = correct / total
        if acc > best_acc: 
            best_acc = acc
            epochs_no_improve = 0
            marker = "Best Record Updated"
        else:
            epochs_no_improve += 1
            marker = f"Patience: {epochs_no_improve}/{patience}"
            
        print(f"Epoch {epoch:03d} | Train Loss: {total_loss/len(train_loader):.4f} | Test Acc: {acc*100:.2f}% (Best: {best_acc*100:.2f}%) | {marker}")
        
        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break
            
    return best_acc

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Environment initialized. Launching Tropical-MorphoGNN on {device}.")
    print("Note: This implementation is a DGL-free, dual-stream architecture aligning point cloud geometry with algebraic topology.\n")
    
    datasets_config = [
        ("ACT-4", "./data/raw/allen_cell_type/swc", "./data/info/ACT_info_swc_10folds.csv", 4),
        ("JML-4", "./data/raw/janelia_mouselight/swc", "./data/info/JML_info_swc_10folds.csv", 4),
        ("BIL-6", "./data/raw/bil/swc", "./data/info/BIL_info_swc_10folds.csv", 6)
    ]
    
    for ds_name, swc_dir, csv_path, num_classes in datasets_config:
        print("\n" + "="*70)
        train_ds = MorphoPointDataset(ds_name, swc_dir, csv_path, phase="train")
        test_ds = MorphoPointDataset(ds_name, swc_dir, csv_path, phase="test")
        
        if len(train_ds) == 0 or len(test_ds) == 0: 
            print(f"Dataset {ds_name} files unlocated or empty. Skipping evaluation.")
            continue
            
        bs = 64
        train_loader = torch.utils.data.DataLoader(train_ds, batch_size=bs, collate_fn=pointcloud_collate_fn, shuffle=True)
        test_loader = torch.utils.data.DataLoader(test_ds, batch_size=bs, collate_fn=pointcloud_collate_fn, shuffle=False)
        
        model = TropicalMorphoGNN(num_classes=num_classes).to(device)
        best_acc = train_and_evaluate(model, train_loader, test_loader, device, epochs=150, patience=25, ds_name=ds_name)
        
        print(f"Final Evaluation for {ds_name}: Tropical-MorphoGNN Maximum Accuracy -> {best_acc*100:.2f}%")

if __name__ == "__main__":
    main()

Environment initialized. Launching Tropical-MorphoGNN on cuda.
Note: This implementation is a DGL-free, dual-stream architecture aligning point cloud geometry with algebraic topology.


[TRAIN] Sampling ACT-4 to 1024 points and extracting mathematical spectra...


100%|██████████| 510/510 [02:48<00:00,  3.03it/s]


[TEST] Sampling ACT-4 to 1024 points and extracting mathematical spectra...


[ACT-4] Epoch 001/150:   0%|          | 0/7 [00:00<?, ?it/s]